# Airline Passenger Satisfaction Prediction

## Feature Engineering

### Objective

Now that I've got a sense of what's in the data from the EDA, this notebook is about getting everything into a shape my models can actually use. Mainly that means encoding the categorical columns, and doing something about the two delay columns since they were basically telling me the same thing (0.97 correlation is redundant).

Here's what I found in the last notebook that's shaping the decisions here:
- satisfaction is fairly balanced (~55/45)
- Inflight entertainment, ease of online booking, and online support were the strongest service-related signals
- Gate location and departure/arrival time convenience barely separated satisfied from dissatisfied passengers at all
- Departure delay and arrival delay were almost perfectly correlated with each other
- Class and travel type both looked like they mattered


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)


### Load the cleaned data

In [2]:
df = pd.read_csv("../data/processed/airline_satisfaction_clean.csv")
df.head()


,satisfaction,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Female,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,2,4,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Male,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,0,2,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Female,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,2,0,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Female,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,3,4,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Female,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,4,3,4,2,2,0,2,4,2,5,0,0.0


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129487 entries, 0 to 129486
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   satisfaction                       129487 non-null  object 
 1   Gender                             129487 non-null  object 
 2   Customer Type                      129487 non-null  object 
 3   Age                                129487 non-null  int64  
 4   Type of Travel                     129487 non-null  object 
 5   Class                              129487 non-null  object 
 6   Flight Distance                    129487 non-null  int64  
 7   Seat comfort                       129487 non-null  int64  
 8   Departure/Arrival time convenient  129487 non-null  int64  
 9   Food and drink                     129487 non-null  int64  
 10  Gate location                      129487 non-null  int64  
 11  Inflight wifi service              1294

### Encode the target

In [4]:
df["satisfaction"] = df["satisfaction"].map({"dissatisfied": 0, "satisfied": 1})
df["satisfaction"].value_counts()


1    70882
0    58605
Name: satisfaction, dtype: int64

### Encode the binary categorical columns

Gender, Customer Type, and Type of Travel only have two values each, so a simple 0/1 map works fine here, no need for one-hot encoding these.

In [5]:
df["Gender"] = df["Gender"].map({"Female": 0, "Male": 1})
df["Customer Type"] = df["Customer Type"].map({"disloyal Customer": 0, "Loyal Customer": 1})
df["Type of Travel"] = df["Type of Travel"].map({"Personal Travel": 0, "Business travel": 1})

df[["Gender", "Customer Type", "Type of Travel"]].head()


,Gender,Customer Type,Type of Travel
0,0,1,0
1,1,1,0
2,0,1,0
3,0,1,0
4,0,1,0


### Encode Class

Class has three levels (Eco, Eco Plus, Business), and from the EDA it wasn't a clean, even step from Eco to Eco Plus to Business. Eco Plus was much closer to a toss-up rather than sitting neatly in the middle. Since I can't be sure the relationship is linear, I'm one-hot encoding it instead of just mapping it to 0/1/2, so I'm not forcing an assumption
about ordering that the data didn't clearly support.

In [6]:
df = pd.get_dummies(df, columns=["Class"], prefix="class", drop_first=True)
df.head()


,satisfaction,Gender,Customer Type,Age,Type of Travel,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes,class_Eco,class_Eco Plus
0,1,0,1,65,0,265,0,0,0,2,2,4,2,3,3,0,3,5,3,2,0,0.0,1,0
1,1,1,1,47,0,2464,0,0,0,3,0,2,2,3,4,4,4,2,3,2,310,305.0,0,0
2,1,0,1,15,0,2138,0,0,0,3,2,0,2,2,3,3,4,4,4,2,0,0.0,1,0
3,1,0,1,60,0,623,0,0,0,3,3,4,3,1,1,0,1,4,1,3,0,0.0,1,0
4,1,0,1,70,0,354,0,0,0,3,4,3,4,2,2,0,2,4,2,5,0,0.0,1,0


### Combine the delay columns

Departure and arrival delay were correlated at 0.97 in the EDA, which makes sense since a late departure usually just means a late arrival too. Keeping both as separate features mostly just adds redundant, near-duplicate information, so I'm combining them into a single total_delay feature instead of carrying two almost-identical columns forward.

In [7]:
df["total_delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]

df = df.drop(columns=["Departure Delay in Minutes", "Arrival Delay in Minutes"])

df[["total_delay"]].describe()


,total_delay
count,129487.000000
mean,29.734514
std,75.732722
min,0.000000
25%,0.000000
50%,2.000000
75%,24.000000
max,3176.000000


### Add a composite service score

On top of keeping all 14 individual service ratings (so the model can still pick up on which specific categories matter most), I'm adding one new feature that averages all of them into a single "overall service experience" score. The idea is this might help a model like logistic regression pick up on the general pattern more easily, even though a tree-based model probably won't need it as much.

In [8]:
service_cols = [
    "Seat comfort", "Departure/Arrival time convenient", "Food and drink",
    "Gate location", "Inflight wifi service", "Inflight entertainment",
    "Online support", "Ease of Online booking", "On-board service",
    "Leg room service", "Baggage handling", "Checkin service",
    "Cleanliness", "Online boarding"
]

df["avg_service_score"] = df[service_cols].mean(axis=1)

df[["avg_service_score"]].describe()


,avg_service_score
count,129487.000000
mean,3.310156
std,0.672716
min,1.071429
25%,2.857143
50%,3.357143
75%,3.785714
max,5.000000


### Separate features and target

In [9]:
target = df["satisfaction"]
features = df.drop(columns=["satisfaction"])

print("Feature matrix shape:", features.shape)
print("Target shape:", target.shape)


Feature matrix shape: (129487, 23)
Target shape: (129487,)


### Scaled version of the numeric features

Tree-based models don't need scaling, but if I want to try logistic regression too, it does. Saving both versions now so I don't have to redo this step later depending on which model ends up working best.

In [10]:
numeric_cols = [
    "Age", "Flight Distance", "total_delay", "avg_service_score"
] + service_cols

scaler = StandardScaler()
features_scaled = features.copy()
features_scaled[numeric_cols] = scaler.fit_transform(features[numeric_cols])

features_scaled.head()


,Gender,Customer Type,Age,Type of Travel,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,class_Eco,class_Eco Plus,total_delay,avg_service_score
0,0,1,1.691495,0,-1.671090,-2.037943,-1.958042,-1.975658,-0.758380,-0.947223,0.457857,-1.163548,-0.36166,-0.366038,-2.698079,-0.601358,1.316301,-0.612919,-1.041526,1,0,-0.392626,-1.841389
1,1,1,0.500825,0,0.470348,-2.037943,-1.958042,-1.975658,0.007368,-2.463800,-1.028077,-1.163548,-0.36166,0.420898,0.397718,0.263333,-1.063601,-0.612919,-1.041526,0,0,7.728070,-1.841389
2,0,1,-1.615922,0,0.152882,-2.037943,-1.958042,-1.975658,0.007368,-0.947223,-2.514012,-1.163548,-1.12761,-0.366038,-0.376231,0.263333,0.523000,0.255379,-1.041526,1,0,-0.392626,-1.841389
3,0,1,1.360753,0,-1.322461,-2.037943,-1.958042,-1.975658,0.007368,-0.188935,0.457857,-0.398040,-1.89356,-1.939912,-2.698079,-2.330739,0.523000,-2.349514,-0.271477,1,0,-0.392626,-2.372288
4,0,1,2.022237,0,-1.584420,-2.037943,-1.958042,-1.975658,0.007368,0.569353,-0.285110,0.367469,-1.12761,-1.152975,-2.698079,-1.466048,0.523000,-1.481216,1.268621,1,0,-0.392626,-1.629030


### Correlation check on the new features

In [11]:
check_df = features.copy()
check_df["satisfaction"] = target

new_features = ["total_delay", "avg_service_score"]
check_df[new_features + ["satisfaction"]].corr()["satisfaction"].sort_values(ascending=False)


satisfaction         1.000000
avg_service_score    0.505979
total_delay         -0.078029
Name: satisfaction, dtype: float64

### Save the engineered datasets

In [12]:
output_unscaled = features.copy()
output_unscaled["satisfaction"] = target
output_unscaled.to_csv("../data/processed/airline_satisfaction_features.csv", index=False)

output_scaled = features_scaled.copy()
output_scaled["satisfaction"] = target
output_scaled.to_csv("../data/processed/airline_satisfaction_features_scaled.csv", index=False)

print("Saved:")
print(" - ../data/processed/airline_satisfaction_features.csv", output_unscaled.shape)
print(" - ../data/processed/airline_satisfaction_features_scaled.csv", output_scaled.shape)


Saved:
 - ../data/processed/airline_satisfaction_features.csv (129487, 24)
 - ../data/processed/airline_satisfaction_features_scaled.csv (129487, 24)


## Summary

- **Encoding:** Gender, Customer Type, and Type of Travel mapped to 0/1 without any issues. Class got one-hot encoded into class_Eco and class_Eco Plus (Business is the dropped baseline), since it didn't look clearly ordinal in the EDA.

- **Delay columns combined:** total_delay replaces the two separate delay columns. It's got a similarly wild distribution to what I saw before combining them, median of just 2 minutes, but a max of 3176 minutes (over 52 hours combined). Makes sense, since I'm just adding together two already heavily-skewed columns.

- **New feature - avg_service_score:** averages out to 3.31 across all 14 service ratings, ranging from 1.07 to a perfect 5.0. This is the one I was most curious about.

- **Correlation check:** avg_service_score correlates with satisfaction at **0.51**, which puts it right up with inflight entertainment (0.52), the strongest individual service rating from the EDA. So averaging all 14 ratings together didn't dilute the signal at all if anything it's basically as strong as the single best individual feature, which is a nice sign that it's capturing something real about overall experience.

- total_delay came in at **-0.078**, which lines up almost exactly with what the two individual delay columns showed in the EDA (-0.07 and -0.08). So combining them didn't lose any signal, and confirms again that delays just aren't a big driver of satisfaction in this dataset compared to the service experience.

- **Outputs:** both a scaled and unscaled version saved, (23 features + the target).

**Next step:** train a couple of candidate models and compare them properly.
